In [ ]:
import os

# Single GPU — must be set before JAX initialises
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import jax
print("JAX devices:", jax.devices())   # should show exactly 1 GPU

In [2]:
!pip install -q git+https://github.com/winstonsmith1897/DantinoX.git#egg=dantinox[all]

DEPRECATION: git+https://github.com/winstonsmith1897/DantinoX.git#egg=dantinox[all] contains an egg fragment with a non-PEP 508 name pip 25.0 will enforce this behaviour change. A possible replacement is to use the req @ url syntax, and remove the egg fragment. Discussion can be found at https://github.com/pypa/pip/issues/11617


In [ ]:
import json
import dantinox as dx
import jax.numpy as jnp
import numpy as np
from flax import nnx

In [ ]:
# ==============================================================================
# 1.  UNSUPERVISED — SimCSE with the stock Trainer
# ==============================================================================
#
# Uses wikitext-2 (HuggingFace) — no local files needed.
# The Trainer downloads and tokenises it once, then caches a .npy mmap file.
#
# dropout=0.1 is REQUIRED: SimCSE encodes the same window twice with different
# dropout masks. With dropout=0.0 the two views are identical and the loss
# collapses (InfoNCE diagonal = 1, off-diagonal = 1 → no gradient).

cfg = dx.ModelConfig(
    dim=128, n_heads=4, head_size=32, num_blocks=4,
    vocab_size=4_096,   # will be overridden by the tokenizer vocab size at fit time
    causal=False,       # bidirectional encoder
    dropout=0.1,        # REQUIRED for SimCSE
    max_context=128,
)
paradigm = dx.EmbedderParadigm(cfg, pooling="mean", temperature=0.05)
print(paradigm)

In [ ]:
# No local corpus needed — the Trainer pulls wikitext-2 from HuggingFace,
# trains a BPE tokenizer on the fly, and caches everything as a .npy mmap.

train_cfg = dx.TrainingConfig(
    # ── HuggingFace dataset ──────────────────────────────────────────────────
    dataset_source="huggingface",
    dataset_name="wikitext",
    dataset_config="wikitext-2-raw-v1",
    dataset_text_field="text",
    dataset_split="train",
    tokenizer_type="bpe",
    # ── training ─────────────────────────────────────────────────────────────
    lr=3e-4,
    epochs=3,
    batch_size=64,
    val_frac=0.05,
    warmup_steps=100,
    max_train_tokens=500_000,   # cap for a quick Colab demo
)
print(train_cfg)

In [ ]:
run_dir = dx.train(paradigm, training_config=train_cfg)
print("Run dir:", run_dir)

In [ ]:
# ==============================================================================
# 2.  SUPERVISED — (anchor, positive) pairs with EmbedderTrainer
# ==============================================================================
#
# We build pairs from wikitext-2 directly: consecutive sentence pairs from the
# same paragraph are semantically related → good positives for InfoNCE.
# No separate labelled dataset required.

from datasets import load_dataset

wiki = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

def extract_pairs(dataset, max_pairs: int = 2_000) -> list[tuple[str, str]]:
    """Consecutive non-empty sentences in the same paragraph = positive pairs."""
    pairs = []
    for row in dataset:
        text = row["text"].strip()
        if not text or text.startswith(" ="):   # skip section headers
            continue
        sentences = [s.strip() for s in text.split(".") if len(s.strip()) > 30]
        for i in range(len(sentences) - 1):
            pairs.append((sentences[i], sentences[i + 1]))
            if len(pairs) >= max_pairs:
                return pairs
    return pairs

pairs = extract_pairs(wiki, max_pairs=2_000)
print(f"{len(pairs)} pairs extracted")
print("Example:", pairs[0])

In [ ]:
# Reuse the tokenizer trained by the unsupervised step above.
# Load it from the run directory so both modes share the same vocabulary.
from dantinox.utils.tokenizer import load_tokenizer_from_file
tok = load_tokenizer_from_file(f"{run_dir}/tokenizer.json")

cfg_sup = dx.ModelConfig(
    dim=128, n_heads=4, head_size=32, num_blocks=4,
    vocab_size=tok.vocab_size,
    causal=False,
    dropout=0.1,
    max_context=128,
)
paradigm_sup = dx.EmbedderParadigm(cfg_sup, pooling="mean", temperature=0.05)

trainer = dx.EmbedderTrainer(
    paradigm_sup, tok,
    dx.TrainingConfig(lr=2e-4, epochs=5, batch_size=32),
)
run_dir_sup = trainer.fit_pairs(pairs, run_dir="runs/embedder_supervised")
print("Run dir:", run_dir_sup)

In [ ]:
# ==============================================================================
# 3.  INFERENCE — Embedder
# ==============================================================================

embedder = dx.Embedder.from_run(run_dir_sup)
print(f"dim = {embedder.dim}")

# Use real sentences from the wikitext test split
wiki_test = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
sentences = [
    row["text"].strip()
    for row in wiki_test
    if len(row["text"].strip()) > 60 and not row["text"].strip().startswith(" =")
][:20]

queries   = sentences[:3]
documents = sentences[3:13]

q_vecs = embedder.embed(queries)    # [3, D]
d_vecs = embedder.embed(documents)  # [10, D]

# Cosine similarity — L2-normalised vectors → dot product == cosine
sim = q_vecs @ d_vecs.T             # [3, 10]
for i, q in enumerate(queries):
    best = int(sim[i].argmax())
    print(f"\nQ: {q[:80]!r}")
    print(f"→  {documents[best][:80]!r}  (score={sim[i, best]:.3f})")

In [ ]:
# ==============================================================================
# 4.  FAISS — fast vector store
# ==============================================================================
# !pip install -q faiss-gpu   # or faiss-cpu

import faiss

# IndexFlatIP = inner product (== cosine similarity for L2-normalised vectors)
index = faiss.IndexFlatIP(embedder.dim)
index.add(d_vecs.astype(np.float32))
print(f"Index: {index.ntotal} documents")

# Retrieve top-2 for each query
D, I = index.search(q_vecs.astype(np.float32), k=2)
for i, q in enumerate(queries):
    print(f"\nQ: {q!r}")
    for rank, (score, doc_idx) in enumerate(zip(D[i], I[i])):
        print(f"  [{rank+1}] ({score:.3f}) {documents[doc_idx]!r}")

In [ ]:
# ==============================================================================
# 5a.  LANGCHAIN — drop-in integration
# ==============================================================================
# !pip install -q langchain langchain-community

from langchain_community.vectorstores import FAISS as LC_FAISS

lc_embed = embedder.as_langchain_embeddings()
store = LC_FAISS.from_texts(documents, embedding=lc_embed)

results = store.similarity_search("How does JAX work?", k=2)
for r in results:
    print(r.page_content)

In [ ]:
# ==============================================================================
# 5b.  CHROMADB — drop-in integration
# ==============================================================================
# !pip install -q chromadb

import chromadb

client = chromadb.Client()
col = client.create_collection(
    "dantinox_docs",
    embedding_function=embedder.as_chroma_fn(),
)
col.add(
    documents=documents,
    ids=[str(i) for i in range(len(documents))],
)

results = col.query(query_texts=["What is a transformer?"], n_results=2)
for doc in results["documents"][0]:
    print(doc)

In [ ]:
# ==============================================================================
# 6.  FINE-TUNING a pretrained model (AR / Discrete → Embedder)
# ==============================================================================
#
# Any DantinoX model can be fine-tuned as an embedder without retraining
# from scratch.  Pass model= to fit_pairs() to inject the pretrained weights.

pretrained_run = "runs/my_discrete_run"   # ← your existing run

pretrained_cfg   = dx.ModelConfig.from_yaml(f"{pretrained_run}/config.yaml")
pretrained_model = dx.load(pretrained_run)

# Wrap in EmbedderParadigm — same architecture, new contrastive loss
ft_paradigm = dx.EmbedderParadigm(pretrained_cfg, pooling="mean", temperature=0.05)

ft_trainer = dx.EmbedderTrainer(
    ft_paradigm, tok,
    dx.TrainingConfig(lr=5e-5, epochs=5, batch_size=16),  # low lr for fine-tuning
)

# model= injects pretrained weights instead of starting from scratch
run_dir_ft = ft_trainer.fit_pairs(
    pairs,
    model=pretrained_model,
    run_dir="runs/embedder_finetuned",
)
print("Fine-tuning done →", run_dir_ft)

embedder_ft = dx.Embedder.from_run(run_dir_ft)
print("dim:", embedder_ft.dim)